### 1. Preprocess: Segment engagement, disengagement intervals

In [1]:
import pandas as pd
import opensmile
import librosa
import numpy as np
import os

In [2]:
# Read annotation files
def read_annotation_file(eng_df):
  eng_df.rename(columns={'Timestamp': 'timestamp'}, inplace=True)
  eng_df.drop(columns=['Unnamed: 2'], inplace=True)
  # print(eng_df.head())
  return eng_df  


In [3]:
# Extract engaged and disengaged timestamps -> Output: [(begin, end, label), ...]
def get_interval_label(eng_df):
    all_intervals = []
    current_label = eng_df.loc[0, 'C_Engagement']
    current_start_time = float(eng_df.loc[0, 'timestamp'])
    for i in range(1, len(eng_df)):
        last_timestamp = float(eng_df.loc[i-1, 'timestamp'])
        label = eng_df.loc[i, 'C_Engagement']
        if label != current_label:
            all_intervals.append((current_start_time, last_timestamp, current_label))
            current_start_time = float(eng_df.loc[i, 'timestamp'])
            current_label = label
    # Append the last interval
    all_intervals.append((current_start_time, float(eng_df.loc[len(eng_df)-1, 'timestamp']), current_label))
    print(all_intervals)
    return all_intervals

In [4]:
# Extract engaged and disengaged timestamps -> Output: [(begin, end, label), ...]
def get_interval_label_P5(eng_df):

    engagement_intervals = []

    for _, row in eng_df.iterrows():

        start_time = float(row["start_time"])
        end_time = float(row["end_time"])
        label = row["label"]

        engagement_intervals.append(
            (start_time, end_time, label)
        )

    return engagement_intervals

In [5]:
# Get average & total lengths of intervals for engaged vs disengaged states
def get_avg_length_intervals(intervals):
    lengths = {'engaged': [], 'disengaged': []}
    for i in intervals:
      start_time, end_time, label = i
      if label == 'disengaged':
        lengths['disengaged'].append(end_time - start_time)
      elif label == 'engaged':
        lengths['engaged'].append(end_time - start_time)    
    print(len(lengths['engaged']), len(lengths['disengaged']))
    engaged_avg = np.mean(lengths['engaged'])
    disengaged_avg = np.mean(lengths['disengaged'])
    engaged_total = np.sum(lengths['engaged'])
    disengaged_total = np.sum(lengths['disengaged'])
    return {'engaged_avg': float(engaged_avg), 'disengaged_avg': float(disengaged_avg), 'engaged_total': float(engaged_total), 'disengaged_total': float(disengaged_total)}

### 2. Extract Audio Features using eGeMAPs

### 2-1. Low-Level Descriptors
Low-Level Descriptors (LLD) are 25 acoustic measurements extracted in frame-level. 

c.f. Functionals: "functionals (statistical, polynomial regression coefficients, and transformations) listed in table 2 can be applied, e.g. to low-level features. This is a common technique, e.g. in emotion recognition and Music Information Retrieval" ([paper](https://mediatum.ub.tum.de/doc/1082431/1082431.pdf)). 

Read: "The eGeMAPS set consist of 88 features derived from a base of 25 low-level acoustic descriptors(LLDs) extracted at a frame-level. The 18 LLDs are pitch, jitter, centerfrequencies of the first 3 formants, bandwidths of the first 3 formants, relative energies of the first 3 formants, loudness, shimmer, alpha ratio, harmonics-to-noise ratio, Hammarberg index, spectral slope, harmonic differences and the first four MFCCs (1–4). The arithmetic mean and coefficient of variation (standard deviation normalized by the arithmetic mean) are applied as functionals to all LLDs. Additional 8 functionalsare applied to pitch and loudness. For most parameters, the functionals are applied only in voiced regions. In the case of a few parameters like alpha ratio, functionals computed over all unvoiced frames are alsoincluded. Additionally, 6 temporal features (like rate of loudness peaks) and equivalent sound level are appended. Altogether 88 features are computed for every input speech signal. " ([paper](https://www.sciencedirect.com/science/article/pii/S0167639324000128))

In [6]:
# Add labels for each frame based on the intervals
def get_label_for_time(start_sec, end_sec, intervals):
  for interval in intervals:
    interval_start, interval_end, label = interval
    if start_sec >= interval_start and end_sec <= interval_end:
      return label
  return 'unknown'  # if no matching interval is found

In [7]:
# Extract eGeMAPS LLD features for the entire audio
def extract_lld_features(audio_path, all_intervals):
    # Initialize OpenSMILE with the eGeMAPS feature set (v02)
    lld_smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.LowLevelDescriptors,
    )
    # print('lld features:', lld_smile.feature_names)
    print('Number of lld features:', len(lld_smile.feature_names))

    # reference: https://librosa.org/doc/main/generated/librosa.load.html
    # y: audio time series, sr: sampling rate (sr=None: use original sampling rate of the file)
    y, sr = librosa.load(path=audio_path, sr=None)
    # y: sr * length of audio in seconds
    # sr: how many samples per second (e.g., 16000 means 16000 samples in 1 second)
    print('y.shape:', y.shape, 'sr:', sr)
    
    # reference: https://audeering.github.io/opensmile-python/api/opensmile.Smile.html#process-signal
    # Extract eGeMAPS LLD features for the entire audio
    lld_features = lld_smile.process_signal(y, sr)
    # print('Number of frames:', len(lld_features))  # note: default frame step: 10ms (0.01s) -> length of audio in seconds * 100
    
    # Add start and end times (in seconds)
    lld_features['start_sec'] = lld_features.index.get_level_values('start').total_seconds()
    lld_features['end_sec'] = lld_features.index.get_level_values('end').total_seconds()
    # print(lld_features[['start_sec', 'end_sec']].head())

    # Remove start and end from index
    lld_features.reset_index(drop=True, inplace=True)
    # print(lld_features.head())
    
    # Add labels for each frame based on the intervals
    lld_features['label'] = lld_features.apply(lambda row: get_label_for_time(row['start_sec'], row['end_sec'], all_intervals), axis=1)
    # print('Renamed index:', lld_features[['start_sec', 'end_sec', 'label']].head())
    # print('Number of frames:', len(lld_features))
    
    return lld_features


### 2-2. Functionals
Using the engaged/disengaged intervals, we extract functional features for each of the intervals.

In [8]:
# Extract functional features
def extract_functional_features(audio_path, all_intervals):
    functional_features = pd.DataFrame()
    
    # Initialize OpenSMILE with the eGeMAPS feature set (v02)
    f_smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    # print('functional features:', f_smile.feature_names)
    print('Number of functional features:', len(f_smile.feature_names))
    
    # reference: https://librosa.org/doc/main/generated/librosa.load.html
    # y: audio time series, sr: sampling rate (sr=None: use original sampling rate of the file)
    y, sr = librosa.load(path=audio_path, sr=None)
    # y: sr * length of audio in seconds
    # sr: how many samples per second (e.g., 16000 means 16000 samples in 1 second)
    print('y.shape:', y.shape, 'sr:', sr)
    
    for start_time, end_time, label in all_intervals:
        start_idx = int(start_time * sr)
        end_idx = int(end_time * sr)
        segment = y[start_idx:end_idx]
        features = f_smile.process_signal(segment, sr)
        features['start_time'] = start_time
        features['end_time'] = end_time
        features['label'] = label
        functional_features = pd.concat([functional_features, features], ignore_index=True)
    return functional_features
    

### 4. Run on all files

In [9]:
p_s = [(5,2), (5,7), (5,8), (5,10), (5,13), (7,5), (7,6), (7,7), (7,8), (7,16), (7,17), (7,18), (7,29), (9,'3-1'), (9,'3-2'), (9,4), (9,9), (9,15), (11,2), (11,4), (11,8), (11,9), (11,11), (11,15), (11,'16-2'), (11,19), (11,'22-2'), (12,'2-2'), (12,3), (12,6), (12,8), (12,10), (17,2), (17,3), (17,5), (17,6), (18,3), (18,4), (18,5), (18,7), (18,8), (18,9), (18,10), (18,11), (18,12), (18,13), (18,15), (18,17), (18,18), (18,19),(18,20)]	

for (p, s) in p_s:
  print(f"Processing participant {p}, session {s}...")
  eng_df_path = f"eGeMAPs_features/p{p}/functional_p{p}-s{s}.csv"
  audio_path = f"/media/interaction-1/One Touch/ASD_Dataset/all-audios/p{p}-s{s}.wav"
  
  # eng_df = read_annotation_file(pd.read_csv(eng_df_path))
  eng_df = pd.read_csv(eng_df_path)
  engagement_intervals = get_interval_label_P5(eng_df)
  avg = get_avg_length_intervals(engagement_intervals)
  print(avg)
  
  # Extract & Save the LLD features to a CSV file
  # lld_features = extract_lld_features(audio_path, all_intervals)
  # lld_features.to_csv(f"/Users/youngsuh-hong/ys_files/USC/V2_audio_analysis/asd-audio-clip/eGeMAPs-features_denoised/lld_" + f"p{p}-s{s}.csv", index=True)
  
  # Extract & Save the functional features to a CSV file
  functional_features = extract_functional_features(audio_path, engagement_intervals)
  
  output_dir = os.path.join(
      "output-eGeMAPs-raw",
      "functionals",
      f"p{p}"
  ) 

  os.makedirs(output_dir, exist_ok=True)

  output_file = os.path.join(
      output_dir,
      f"functionals_p{p}-s{s}.csv"
  )

  functional_features.to_csv(
      output_file,
      index=False
  )
  print("================================")

Processing participant 5, session 2...
32 32
{'engaged_avg': 38.30274999999999, 'disengaged_avg': 1.9662187500000103, 'engaged_total': 1225.6879999999996, 'disengaged_total': 62.91900000000033}
Number of functional features: 88


/home/interaction-1/anaconda3/envs/gemaps_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


y.shape: (21604320,) sr: 16000
Processing participant 5, session 7...
31 26
{'engaged_avg': 40.803806451612914, 'disengaged_avg': 3.7997307692307625, 'engaged_total': 1264.9180000000003, 'disengaged_total': 98.79299999999982}
Number of functional features: 88
y.shape: (21922720,) sr: 16000
Processing participant 5, session 8...
52 45
{'engaged_avg': 24.76386538461539, 'disengaged_avg': 55.525533333333364, 'engaged_total': 1287.7210000000002, 'disengaged_total': 2498.6490000000013}
Number of functional features: 88
y.shape: (61501920,) sr: 16000
Processing participant 5, session 10...
35 35
{'engaged_avg': 24.512942857142864, 'disengaged_avg': 7.470371428571434, 'engaged_total': 857.9530000000002, 'disengaged_total': 261.4630000000002}
Number of functional features: 88
y.shape: (18061920,) sr: 16000
Processing participant 5, session 13...
32 34
{'engaged_avg': 29.58715625000002, 'disengaged_avg': 11.4275294117647, 'engaged_total': 946.7890000000007, 'disengaged_total': 388.5359999999998